# Stage 4 — AIS-Derived Departure and ETA-Based Berth Availability

Keberangkatan hanya dibentuk dari bukti AIS. Dermaga tujuan dinilai pada waktu
ETA kapal, bukan pada waktu keberangkatan.

In [ ]:
#@title Shared MFAR paths and stage initialization
from pathlib import Path
from datetime import datetime, timezone
import os, sys, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# Locate the code repository only; all simulation I/O paths are resolved by
# src.mfar_paths through MFAR_GDRIVE_ROOT or the mounted/synchronized Drive.
_code_candidates = [Path.cwd(), Path.cwd().parent]
if os.environ.get("MFAR_CODE_ROOT"):
    _code_candidates.insert(0, Path(os.environ["MFAR_CODE_ROOT"]))
_code_candidates.extend([
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/MFAR_Modular_Colab_Pipeline"),
])
for _candidate in _code_candidates:
    if (_candidate / "src" / "mfar_paths.py").is_file():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "Modul src/mfar_paths.py tidak ditemukan. Jalankan notebook dari repository "
        "atau tetapkan MFAR_CODE_ROOT ke folder repository."
    )

from src.mfar_paths import (
    AIS_RAW_PATH, VEHICLE_ARRIVAL_PATH, DATA_RAW_DIR, CONFIG_DIR,
    STAGE_OUTPUT_DIR, STAGE_01_DIR, STAGE_02_DIR, STAGE_03_DIR,
    STAGE_04_DIR, STAGE_05_DIR, STAGE_06_DIR, STAGE_07_DIR,
    validate_csv_input, validate_raw_inputs, validate_writable_directory,
    write_execution_metadata,
)

_MFAR_STARTED_AT = datetime.now(timezone.utc)

NOTEBOOK_NAME = "04_No_Intervention_Forecast.ipynb"
RAW, CFG, STAGE = DATA_RAW_DIR, CONFIG_DIR, STAGE_OUTPUT_DIR
validate_writable_directory(STAGE_04_DIR, NOTEBOOK_NAME, 4)
print("Input stage:", STAGE_03_DIR)
print("Vehicle arrivals:", VEHICLE_ARRIVAL_PATH)
print("Output folder:", STAGE_04_DIR)


In [ ]:
state=validate_csv_input(STAGE_03_DIR/"03_input_state_enhanced.csv", ["grid_time","mmsi","operational_status","origin","destination","nearest_distance_nm","is_at_berth","berth_episode_id"], NOTEBOOK_NAME, 4)
state["grid_time"]=pd.to_datetime(state["grid_time"],errors="coerce")
for c in ["berth_entry_time","predicted_berth_release_time"]:
    if c in state:
        state[c]=pd.to_datetime(state[c],errors="coerce")

rates=validate_csv_input(VEHICLE_ARRIVAL_PATH, ["port_id","time_start","time_end","car_arrival_rate_30min","motorcycle_arrival_rate_30min"], NOTEBOOK_NAME, 4)
profiles=pd.read_csv(CFG/"vessel_profiles.csv")
berths=pd.read_csv(CFG/"terminal_berths.csv")

SIM_DATE=pd.to_datetime(
    os.environ.get("MFAR_SIM_DATE", str(state["grid_time"].dt.date.min()))
).date()
START=pd.Timestamp(f"{SIM_DATE} 07:00")
END=pd.Timestamp(f"{SIM_DATE} 23:55")
TIMES=pd.date_range(START,END,freq="5min")
MOTOR_CE=0.25
DEFAULT_CAP=float(profiles["vehicle_capacity_ce"].median())

In [ ]:
# ============================================================
# 1. AIS-BASED DEPARTURE DETECTION
# ============================================================
# A departure is accepted only when all evidence is present:
# - vessel was previously at berth,
# - current state is departing/sailing,
# - speed rises above threshold,
# - distance from origin berth increases,
# - movement persists in the following AIS/grid point.

day=state[state["grid_time"].dt.date==SIM_DATE].copy()
day=day.sort_values(["mmsi","grid_time"])

day["prev_status"]=day.groupby("mmsi")["operational_status"].shift()
day["next_status"]=day.groupby("mmsi")["operational_status"].shift(-1)
day["prev_sog"]=day.groupby("mmsi")["sog"].shift()
day["next_sog"]=day.groupby("mmsi")["sog"].shift(-1)
day["prev_nearest_distance_nm"]=day.groupby("mmsi")["nearest_distance_nm"].shift()
day["next_nearest_distance_nm"]=day.groupby("mmsi")["nearest_distance_nm"].shift(-1)

day["ais_departure_candidate"]=(
    day["prev_status"].astype(str).str.startswith("AT_BERTH")
    & day["operational_status"].astype(str).str.match(r"DEPARTING|SAILING")
    & pd.to_numeric(day["sog"],errors="coerce").fillna(0).ge(0.8)
    & pd.to_numeric(day["next_sog"],errors="coerce").fillna(0).ge(0.8)
    & (
        pd.to_numeric(day["nearest_distance_nm"],errors="coerce")
        >
        pd.to_numeric(day["prev_nearest_distance_nm"],errors="coerce")
    )
    & (
        pd.to_numeric(day["next_nearest_distance_nm"],errors="coerce")
        >=
        pd.to_numeric(day["nearest_distance_nm"],errors="coerce")
    )
)

events=day[day["ais_departure_candidate"]].copy()
events["simulation_time"]=events["grid_time"]
events["event_type"]="AIS_DEPARTURE"
events["capacity_ce"]=pd.to_numeric(
    events.get("vehicle_capacity_ce",DEFAULT_CAP),errors="coerce"
).fillna(DEFAULT_CAP)

# Deduplicate repeated detection for the same departure episode.
events["prev_event_time"]=events.groupby("mmsi")["simulation_time"].shift()
events=events[
    events["prev_event_time"].isna()
    | ((events["simulation_time"]-events["prev_event_time"]).dt.total_seconds()/60>=20)
].copy()
events=events.drop(columns=["prev_event_time"])

# Hard audit: every event must retain AIS evidence.
departure_audit=pd.DataFrame({
    "check":[
        "departure_without_previous_at_berth",
        "departure_without_speed_evidence",
        "departure_without_increasing_distance",
        "duplicate_departure_within_20min"
    ],
    "failed_rows":[
        int((~events["prev_status"].astype(str).str.startswith("AT_BERTH")).sum()),
        int(((events["sog"]<0.8)|(events["next_sog"]<0.8)).sum()),
        int((events["nearest_distance_nm"]<=events["prev_nearest_distance_nm"]).sum()),
        0
    ]
})

In [ ]:
# ============================================================
# 2. EVENT-DRIVEN BASELINE QUEUE
# ============================================================
def arrival_5min(port,t):
    x=rates[rates["port_id"].astype(str).str.upper()==str(port).upper()]
    hhmm=t.strftime("%H:%M")
    for _,r in x.iterrows():
        end=str(r["time_end"])
        valid=(hhmm>=str(r["time_start"])) if end=="24:00" else (
            str(r["time_start"])<=hhmm<end
        )
        if valid:
            return (
                float(r["car_arrival_rate_30min"])/6,
                float(r["motorcycle_arrival_rate_30min"])/6
            )
    return 0.0,0.0

ports=sorted(rates["port_id"].astype(str).str.upper().unique())
q={p:{"car":0.0,"motor":0.0} for p in ports}
queue_rows=[]
event_rows=[]

for t in TIMES:
    for p in ports:
        car,motor=arrival_5min(p,t)
        q[p]["car"]+=car
        q[p]["motor"]+=motor

    due=events[events["simulation_time"].eq(t)]
    for _,e in due.iterrows():
        p=str(e["origin"]).upper()
        if p not in q:
            continue
        available_ce=q[p]["car"]+MOTOR_CE*q[p]["motor"]
        served=min(float(e["capacity_ce"]),available_ce)

        serve_car=min(q[p]["car"],served)
        q[p]["car"]-=serve_car
        rem=served-serve_car
        if rem>0:
            q[p]["motor"]=max(0.0,q[p]["motor"]-rem/MOTOR_CE)

        event_rows.append({
            "simulation_time":t,
            "event_type":"AIS_DEPARTURE",
            "mmsi":e["mmsi"],
            "origin":p,
            "destination":e["destination"],
            "capacity_ce":float(e["capacity_ce"]),
            "served_ce":served,
            "ais_source_time_before":e.get("source_time_before",""),
            "ais_source_time_after":e.get("source_time_after",""),
            "point_source":e.get("point_source",""),
            "confidence_class":e.get("confidence_class","")
        })

    for p in ports:
        ce=q[p]["car"]+MOTOR_CE*q[p]["motor"]
        queue_rows.append({
            "simulation_time":t,
            "port_id":p,
            "queue_car":q[p]["car"],
            "queue_motorcycle":q[p]["motor"],
            "queue_ce":ce,
            "queue_ratio":ce/DEFAULT_CAP
        })

queue=pd.DataFrame(queue_rows)
event_log=pd.DataFrame(event_rows)

In [ ]:
# ============================================================
# 3. OBSERVED/PREDICTED BERTH OCCUPANCY
# ============================================================
occupied=day[day["is_at_berth"].astype(bool)].copy()
intervals=[]

for (mmsi,episode),g in occupied.groupby(["mmsi","berth_episode_id"]):
    g=g.sort_values("grid_time")
    berth_ids=g["current_berth_id"].dropna()
    if berth_ids.empty:
        continue

    observed_end=g["grid_time"].max()+pd.Timedelta(minutes=5)
    predicted_end=(
        g["predicted_berth_release_time"].dropna().max()
        if g["predicted_berth_release_time"].notna().any()
        else observed_end
    )

    intervals.append({
        "mmsi":mmsi,
        "berth_id":str(berth_ids.iloc[0]),
        "port_id":str(g["nearest_port_id"].dropna().iloc[0]),
        "start":g["grid_time"].min(),
        "end":max(observed_end,predicted_end)
    })

intervals=pd.DataFrame(intervals)

# Sailing-time estimate from AIS-derived consecutive berth episodes.
episodes=(
    occupied.groupby(["mmsi","berth_episode_id"])
    .agg(
        port=("nearest_port_id","first"),
        berth_start=("grid_time","min"),
        berth_end=("grid_time","max")
    )
    .reset_index()
    .sort_values(["mmsi","berth_start"])
)

trip_rows=[]
for mmsi,g in episodes.groupby("mmsi"):
    g=g.sort_values("berth_start").reset_index(drop=True)
    for i in range(len(g)-1):
        if g.loc[i,"port"]==g.loc[i+1,"port"]:
            continue
        duration=(g.loc[i+1,"berth_start"]-g.loc[i,"berth_end"]).total_seconds()/60
        if 10<=duration<=120:
            trip_rows.append({
                "mmsi":mmsi,
                "origin":str(g.loc[i,"port"]),
                "destination":str(g.loc[i+1,"port"]),
                "trip_min":duration
            })

trip_durations=pd.DataFrame(trip_rows)
route_median=(
    trip_durations.groupby(["mmsi","origin","destination"])["trip_min"].median()
    if not trip_durations.empty else pd.Series(dtype=float)
)
global_trip=float(trip_durations["trip_min"].median()) if not trip_durations.empty else 45.0

In [ ]:
# ============================================================
# 4. BERTH AVAILABILITY IS EVALUATED AT ETA, NOT DEPARTURE
# ============================================================
forecast=[]
reserved_until={}

for _,e in events.sort_values("simulation_time").iterrows():
    origin=str(e["origin"])
    destination=str(e["destination"])
    key=(e["mmsi"],origin,destination)

    sailing_min=float(route_median.get(key,global_trip))
    approach_min=float(
        pd.to_numeric(pd.Series([e.get("approach_allowance_min",5)]),errors="coerce")
        .fillna(5).iloc[0]
    )

    # ETA is based on AIS departure time + AIS-derived sailing time + approach.
    eta=e["simulation_time"]+pd.Timedelta(minutes=sailing_min+approach_min)

    berth_ids=berths.loc[
        berths["port_id"].astype(str).eq(destination),"berth_id"
    ].astype(str).tolist()
    if not berth_ids:
        continue

    berth_availability={}
    for berth_id in berth_ids:
        # Only occupancy that overlaps the ETA matters.
        overlapping=(
            intervals[
                (intervals["berth_id"].astype(str)==berth_id)
                & (intervals["start"]<=eta)
                & (intervals["end"]>eta)
            ]
            if not intervals.empty else pd.DataFrame()
        )

        observed_or_predicted_release=(
            overlapping["end"].max() if not overlapping.empty else eta
        )

        # Earlier ETA allocations also reserve the berth into the future.
        carried_reservation=reserved_until.get(berth_id,eta)

        berth_availability[berth_id]=max(
            eta,
            observed_or_predicted_release,
            carried_reservation
        )

    selected_berth=min(
        berth_ids,
        key=lambda b:berth_availability[b]
    )
    berth_available_time=berth_availability[selected_berth]
    predicted_wait_min=max(
        0.0,
        (berth_available_time-eta).total_seconds()/60
    )

    service_start=max(eta,berth_available_time)
    turnaround_min=float(
        pd.to_numeric(pd.Series([e.get("turnaround_min",40)]),errors="coerce")
        .fillna(40).iloc[0]
    )
    reserved_until[selected_berth]=(
        service_start+pd.Timedelta(minutes=turnaround_min+5)
    )

    q_origin=queue[
        (queue["simulation_time"]==e["simulation_time"])
        & queue["port_id"].eq(origin)
    ]
    q_dest=queue[
        (queue["simulation_time"]==e["simulation_time"])
        & queue["port_id"].eq(destination)
    ]

    future_events=events[
        (events["origin"].astype(str)==origin)
        & (events["simulation_time"]>e["simulation_time"])
    ].sort_values("simulation_time")

    next_departure=(
        future_events["simulation_time"].iloc[0]
        if len(future_events) else pd.NaT
    )
    minutes_to_next_departure=(
        (next_departure-e["simulation_time"]).total_seconds()/60
        if pd.notna(next_departure) else 180.0
    )
    scheduled_capacity_next_60min=float(
        future_events[
            future_events["simulation_time"]
            <= e["simulation_time"]+pd.Timedelta(minutes=60)
        ]["capacity_ce"].sum()
    )

    origin_ratio=float(q_origin["queue_ratio"].iloc[0]) if len(q_origin) else 0.0
    destination_ratio=float(q_dest["queue_ratio"].iloc[0]) if len(q_dest) else 0.0

    forecast.append({
        **e.to_dict(),
        "ais_departure_time":e["simulation_time"],
        "ais_sailing_time_estimate_min":sailing_min,
        "predicted_eta":eta,
        "assigned_destination_berth":selected_berth,
        "predicted_berth_available_time_at_eta":berth_available_time,
        "predicted_wait_min":predicted_wait_min,
        "berth_availability_at_eta":float(predicted_wait_min<=0),
        "origin_queue_ratio":origin_ratio,
        "destination_queue_ratio":destination_ratio,
        "forecast_confidence":float(e.get("eta_reliability",1.0)),
        "minutes_to_next_departure":minutes_to_next_departure,
        "scheduled_capacity_next_60min":scheduled_capacity_next_60min,
        "capacity_shortfall_next_60min":max(
            0.0,
            origin_ratio*DEFAULT_CAP-scheduled_capacity_next_60min
        )
    })

forecast=pd.DataFrame(forecast)

In [ ]:
# ============================================================
# 5. OUTPUT AND HARD VALIDATION
# ============================================================
S4=STAGE/"stage_04"

queue.to_csv(S4/"04_daily_port_queue_forecast.csv",index=False)
event_log.to_csv(S4/"04_daily_event_log.csv",index=False)
forecast.to_csv(S4/"04_daily_vessel_forecast.csv",index=False)
forecast.to_csv(S4/"04_fuzzy_input.csv",index=False)
departure_audit.to_csv(S4/"04_ais_departure_audit.csv",index=False)

eta_audit=pd.DataFrame({
    "check":[
        "missing_ais_departure_time",
        "eta_not_after_departure",
        "negative_predicted_wait",
        "berth_available_before_eta",
        "missing_assigned_destination_berth"
    ],
    "failed_rows":[
        int(forecast["ais_departure_time"].isna().sum()) if len(forecast) else 0,
        int((forecast["predicted_eta"]<=forecast["ais_departure_time"]).sum()) if len(forecast) else 0,
        int((forecast["predicted_wait_min"]<0).sum()) if len(forecast) else 0,
        int((
            forecast["predicted_berth_available_time_at_eta"]
            < forecast["predicted_eta"]
        ).sum()) if len(forecast) else 0,
        int(forecast["assigned_destination_berth"].isna().sum()) if len(forecast) else 0
    ]
})
eta_audit.to_csv(S4/"04_eta_berth_forecast_audit.csv",index=False)

summary=pd.DataFrame({
    "metric":[
        "ais_departures_used",
        "forecast_rows",
        "max_queue_ce",
        "mean_predicted_wait_min",
        "max_predicted_wait_min",
        "unavailable_at_eta_rows",
        "available_at_eta_rows"
    ],
    "value":[
        len(event_log),
        len(forecast),
        float(queue["queue_ce"].max()) if len(queue) else np.nan,
        float(forecast["predicted_wait_min"].mean()) if len(forecast) else np.nan,
        float(forecast["predicted_wait_min"].max()) if len(forecast) else np.nan,
        int((forecast["berth_availability_at_eta"]==0).sum()) if len(forecast) else 0,
        int((forecast["berth_availability_at_eta"]==1).sum()) if len(forecast) else 0
    ]
})
summary.to_csv(S4/"04_ais_eta_berth_summary.csv",index=False)

display(departure_audit)
display(eta_audit)
display(summary)

In [ ]:
#@title Execution metadata and saved-artifact report
_stage_dir = STAGE_04_DIR
_saved_files = sorted(_stage_dir.glob("04_*"))
write_execution_metadata(
    stage=4, notebook=NOTEBOOK_NAME, started_at=_MFAR_STARTED_AT,
    input_paths=[STAGE_03_DIR/'03_input_state_enhanced.csv', VEHICLE_ARRIVAL_PATH, CFG/'vessel_profiles.csv', CFG/'terminal_berths.csv'],
    input_rows={"state": len(state)},
    output_rows={"forecast": len(forecast)},
    output_files=_saved_files,
)
